In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
import joblib
import json

# Importações do Scikit-learn (necessárias para o joblib carregar os objetos)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.cluster import DBSCAN, KMeans, MiniBatchKMeans, HDBSCAN
from sklearn.semi_supervised import LabelSpreading, SelfTrainingClassifier
from xgboost import XGBClassifier

# Métricas e Visualização
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score
)

# Widgets Interativos
import ipywidgets as widgets
from ipywidgets import HBox, Output, VBox
from IPython.display import display, clear_output

# Configurações de visualização
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
sns.set_style('whitegrid')
plt.style.use('fivethirtyeight')

print("Todas as bibliotecas foram importadas com sucesso.")

Todas as bibliotecas foram importadas com sucesso.


In [2]:
# Caminho base para os artefatos de modelo
model_base_path = '..\\..\\data\\preprocessors-pipeline' 

# Dicionário mapeando cada modelo aos seus artefatos salvos
artifact_paths = {
    # Modelos Supervisionados
    'RandomForest': {
        'type': 'pipeline',
        'pipeline': os.path.join(model_base_path, 'RF', 'rf_pipeline_unsw_nb15.joblib'),
        'preprocessor': os.path.join(model_base_path, 'RF', 'rf_preprocessor_unsw_nb15.joblib')
    },
    'XGBoost': {
        'type': 'pipeline_xgb',
        'pipeline': os.path.join(model_base_path, 'XGB', 'xgb_pipeline_unsw_nb15.joblib'),
        'preprocessor': os.path.join(model_base_path, 'XGB', 'xgb_preprocessor_unsw_nb15.joblib')
    },
    'SVM': {
        'type': 'pipeline_svm',
        'pipeline': os.path.join(model_base_path, 'SVM', 'svm_pipeline_unsw_nb15.joblib'),
        'preprocessor': os.path.join(model_base_path, 'SVM', 'svm_preprocessor_unsw_nb15.joblib')
    },
    
    # Modelos Semi-Supervisionados
    'SelfTraining-RF': {
        'type': 'manual',
        'preprocessor': os.path.join(model_base_path, 'SelfTraining-RF', 'st_rf_preprocessor_unsw_nb15.joblib'),
        'model': os.path.join(model_base_path, 'SelfTraining-RF', 'st_rf_model_unsw_nb15.joblib')
    },
    'SelfTraining-XGB': {
        'type': 'manual_xgb',
        'preprocessor': os.path.join(model_base_path, 'SelfTraining-XGB', 'st_xgb_preprocessor_unsw_nb15.joblib'),
        'model': os.path.join(model_base_path, 'SelfTraining-XGB', 'st_xgb_model_unsw_nb15.joblib'),
    },
    'LabelSpreading': {
        'type': 'manual_xgb', # Usa a mesma lógica de predição do ST-XGB (encoder)
        'preprocessor': os.path.join(model_base_path, 'LabelSpreading', 'ls_preprocessor_unsw_nb15.joblib'),
        'model': os.path.join(model_base_path, 'LabelSpreading', 'ls_model_unsw_nb15.joblib'),
        'encoder': os.path.join(model_base_path, 'LabelSpreading', 'ls_encoder_unsw_nb15.joblib')
    },
    'IsolationForest': {
        'type': 'manual_if',
        'preprocessor': os.path.join(model_base_path, 'IF', 'if_preprocessor_unsw_nb15.joblib'),
        'model': os.path.join(model_base_path, 'IF', 'if_model_unsw_nb15.joblib')
    },
    'IsolationForestSubsampled': {
        'type': 'manual_if',
        'preprocessor': os.path.join(model_base_path, 'IF-Subsampled', 'if_subsampled_preprocessor_unsw_nb15.joblib'),
        'model': os.path.join(model_base_path, 'IF-Subsampled', 'if_subsampled_model_unsw_nb15.joblib')
    },
    'KMeans': {
        'type': 'manual_kmeans',
        'preprocessor': os.path.join(model_base_path, 'KMeans', 'kmeans_preprocessor_unsw_nb15.joblib'),
        'model': os.path.join(model_base_path, 'KMeans', 'kmeans_model_unsw_nb15.joblib'),
        'threshold': os.path.join(model_base_path, 'KMeans', 'kmeans_threshold_unsw_nb15.json')
    },
    'MBKMeans': {
        'type': 'manual_kmeans', # Usa a mesma lógica do KMeans
        'preprocessor': os.path.join(model_base_path, 'MBKMeans', 'mbk_preprocessor_unsw_nb15.joblib'),
        'model': os.path.join(model_base_path, 'MBKMeans', 'mbk_model_unsw_nb15.joblib'),
        'threshold': os.path.join(model_base_path, 'MBKMeans', 'mbk_threshold_unsw_nb15.json')
    },
    'HDBSCAN': {
        'type': 'manual_cluster',
        'preprocessor': os.path.join(model_base_path, 'HDBSCAN', 'hdbscan_preprocessor_unsw_nb15.joblib'),
        'model_params': {'min_cluster_size': 15, 'min_samples': 15} # Parâmetros de re-fit
    },
    'DBSCAN': {
        'type': 'manual_cluster',
        'preprocessor': os.path.join(model_base_path, 'DBSCAN', 'dbscan_preprocessor_unsw_nb15.joblib'),
        'model_params': {'eps': 8.5, 'min_samples': 10} # Parâmetros de re-fit
    }
}


print(f"Caminhos para {len(artifact_paths)} tipos de modelo foram definidos.")
print(artifact_paths)

Caminhos para 12 tipos de modelo foram definidos.
{'RandomForest': {'type': 'pipeline', 'pipeline': '..\\..\\data\\preprocessors-pipeline\\RF\\rf_pipeline_unsw_nb15.joblib', 'preprocessor': '..\\..\\data\\preprocessors-pipeline\\RF\\rf_preprocessor_unsw_nb15.joblib'}, 'XGBoost': {'type': 'pipeline_xgb', 'pipeline': '..\\..\\data\\preprocessors-pipeline\\XGB\\xgb_pipeline_unsw_nb15.joblib', 'preprocessor': '..\\..\\data\\preprocessors-pipeline\\XGB\\xgb_preprocessor_unsw_nb15.joblib'}, 'SVM': {'type': 'pipeline_svm', 'pipeline': '..\\..\\data\\preprocessors-pipeline\\SVM\\svm_pipeline_unsw_nb15.joblib', 'preprocessor': '..\\..\\data\\preprocessors-pipeline\\SVM\\svm_preprocessor_unsw_nb15.joblib'}, 'SelfTraining-RF': {'type': 'manual', 'preprocessor': '..\\..\\data\\preprocessors-pipeline\\SelfTraining-RF\\st_rf_preprocessor_unsw_nb15.joblib', 'model': '..\\..\\data\\preprocessors-pipeline\\SelfTraining-RF\\st_rf_model_unsw_nb15.joblib'}, 'SelfTraining-XGB': {'type': 'manual_xgb', 'prep

In [3]:
# --- Carregamento do Dataset de Validação Consolidado ---

# Define o caminho para o arquivo consolidado
data_base_path = '../../data/processed/'
file_name = 'ubuntu_server_dataset_final.csv'
full_path = os.path.join(data_base_path, file_name)

print(f"Carregando dataset de validação consolidado de: {full_path}")

try:
    # Carrega o arquivo CSV único
    df_validation = pd.read_csv(full_path, low_memory=False)

    # Verificando a coluna específica do problema
    coluna_problema = 'response_body_len'
    
    if coluna_problema in df_validation.columns:
        if df_validation[coluna_problema].isnull().values.any():
            print(f"Tratando NaNs encontrados na coluna '{coluna_problema}'...")
            # Preenche NaNs com 0
            df_validation[coluna_problema] = df_validation[coluna_problema].fillna(0)
            
            # Por segurança, também substitui infinitos (se houver)
            df_validation[coluna_problema] = df_validation[coluna_problema].replace([np.inf, -np.inf], 0)
            
            print(f"NaNs e Infs em '{coluna_problema}' preenchidos com 0.")
        else:
            print(f"Coluna '{coluna_problema}' checada, não contém NaNs.")
    else:
        print(f"Aviso: Coluna '{coluna_problema}' não encontrada.")
    
    # Verificação final (este print agora DEVE ser 'False')
    print(f"DataFrame contém NaNs APÓS o tratamento: {df_validation.isnull().values.any()}")
    
    if df_validation.isnull().values.any():
        print("\nATENÇÃO: NaNs ainda persistem! Colunas com NaNs:")
        print(df_validation.isnull().sum()[df_validation.isnull().sum() > 0])
    
    print("--------------------------------------------------\n")
    
    remaining_nans = df_validation.isnull().sum().sum()
    print(f"Total de valores nulos (NaN) após a limpeza final: {remaining_nans}")
    
    if remaining_nans > 0:
        print("\nATENÇÃO: Ainda existem NaNs no DataFrame! Colunas com NaNs:")
        print(df_validation.isnull().sum()[df_validation.isnull().sum() > 0])
    print(f"Arquivo {file_name} lido com sucesso.")
    
    # Embaralha os dados
    df_validation = df_validation.sample(frac=1, random_state=42).reset_index(drop=True)

    # Prepara os dados de validação
    X_val = df_validation.drop(['attack_cat', 'label'], axis=1)
    y_val_text = df_validation['attack_cat']
    y_val_binary = df_validation['label']
    
    # Mapeia todas as categorias de ataque não-normais para 'Ataque' para o relatório binário
    y_val_binary_mapped = np.where(y_val_text == 'Normal', 0, 1)

    print(f"\nDataset de validação consolidado e pronto com {len(df_validation)} linhas.")
    print("\nDistribuição de Classes no Dataset de Validação:")
    print(df_validation['attack_cat'].value_counts(normalize=True) * 100)

except FileNotFoundError:
    print(f"ERRO: Arquivo de dataset consolidado não encontrado em '{full_path}'.")
    print("Por favor, execute o notebook de consolidação primeiro.")
except Exception as e:
    print(f"Ocorreu um erro ao ler o arquivo: {e}")

Carregando dataset de validação consolidado de: ../../data/processed/ubuntu_server_dataset_final.csv
Tratando NaNs encontrados na coluna 'response_body_len'...
NaNs e Infs em 'response_body_len' preenchidos com 0.
DataFrame contém NaNs APÓS o tratamento: False
--------------------------------------------------

Total de valores nulos (NaN) após a limpeza final: 0
Arquivo ubuntu_server_dataset_final.csv lido com sucesso.

Dataset de validação consolidado e pronto com 2063153 linhas.

Distribuição de Classes no Dataset de Validação:
attack_cat
Reconnaissance    95.298652
DoS                2.535149
Exploits           1.705593
Fuzzers            0.371761
Normal             0.071735
Analysis           0.017110
Name: proportion, dtype: float64


In [4]:
# Dicionários globais para armazenar os resultados
all_metrics = []
all_confusion_matrices = {}

def calculate_and_store_metrics(model_name, y_true_bin, y_pred_bin, y_true_txt, y_pred_txt):
    """Calcula e armazena métricas binárias e matrizes de confusão."""
    
    print(f"Calculando métricas para: {model_name}")
    
    # --- Métricas Binárias ---
    # Usamos pos_label=1 (Ataque) para precisão/recall
    metrics_dict = {
        'Modelo': model_name,
        'Accuracy': accuracy_score(y_true_bin, y_pred_bin),
        'Precision (Ataque)': precision_score(y_true_bin, y_pred_bin, pos_label=1, zero_division=0),
        'Recall (Ataque)': recall_score(y_true_bin, y_pred_bin, pos_label=1, zero_division=0),
        'F1-Score (Ataque)': f1_score(y_true_bin, y_pred_bin, pos_label=1, zero_division=0),
        'Precision (Normal)': precision_score(y_true_bin, y_pred_bin, pos_label=0, zero_division=0),
        'Recall (Normal)': recall_score(y_true_bin, y_pred_bin, pos_label=0, zero_division=0),
        'F1-Score (Normal)': f1_score(y_true_bin, y_pred_bin, pos_label=0, zero_division=0)
    }
    all_metrics.append(metrics_dict)
    
    # --- Matrizes de Confusão ---
    
    # Binária
    cm_bin = confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1])
    bin_labels = ['Normal', 'Ataque']
    
    # Multiclasse
    # Para modelos não supervisionados, mapeamos os rótulos verdadeiros para 'Attack'
    y_true_txt_mapped = y_true_txt.copy()
    if model_name in ['IsolationForest', 'KMeans', 'MBKMeans', 'DBSCAN', 'HDBSCAN']:
        y_true_txt_mapped = np.where(y_true_bin == 1, 'Attack', 'Normal')
        
    all_classes = np.unique(np.concatenate((y_true_txt_mapped, y_pred_txt)))
    cm_multi = confusion_matrix(y_true_txt_mapped, y_pred_txt, labels=all_classes)
    
    all_confusion_matrices[model_name] = {
        'binary': (cm_bin, bin_labels),
        'multiclass': (cm_multi, all_classes)
    }

print("Funções auxiliares de métricas definidas.")

Funções auxiliares de métricas definidas.


In [6]:
if 'df_validation' not in locals():
    print("ERRO: Dataset de validação não foi carregado.")
else:
    print("--- Iniciando Loop de Validação dos Modelos ---")
    print(X_val.isnull().values.any())
    print(y_val_text.isnull().values.any())
    print(y_val_binary.isnull().values.any())
    print(df_validation.isnull().values.any())
    
    for model_name, paths in artifact_paths.items():
        print(f"\nValidando: {model_name}...")
        try:
            model_type = paths['type']
            
            # --- Carregar Artefatos ---
            preprocessor = joblib.load(paths['preprocessor']) if 'preprocessor' in paths else None
            model = joblib.load(paths['model']) if 'model' in paths else None
            pipeline = joblib.load(paths['pipeline']) if 'pipeline' in paths else None
            encoder = joblib.load(paths['encoder']) if 'encoder' in paths else None
            
            print(paths)
            
            # --- Processar e Prever ---
            
            # Modelos de Pipeline (RF, SVM)
            if model_type == 'pipeline':
                y_pred_text = pipeline.predict(X_val)
                y_pred_binary = np.where(y_pred_text == 'Normal', 0, 1)

            # Pipeline XGBoost (com encoder)
            elif model_type == 'pipeline_xgb':
                y_pred_encoded = pipeline.predict(X_val)
                y_pred_text = encoder.inverse_transform(y_pred_encoded)
                y_pred_binary = np.where(y_pred_text == 'Normal', 0, 1)

            # Modelos Manuais (SelfTraining-RF)
            elif model_type == 'manual':
                X_val_processed = preprocessor.transform(X_val)
                y_pred_text = model.predict(X_val_processed)
                y_pred_binary = np.where(y_pred_text == 'Normal', 0, 1)

            # Modelos Manuais com Encoder (SelfTraining-XGB, LabelSpreading)
            elif model_type == 'manual_xgb':
                X_val_processed = preprocessor.transform(X_val)
                y_pred_encoded = model.predict(X_val_processed)
                y_pred_text = encoder.inverse_transform(y_pred_encoded)
                y_pred_binary = np.where(y_pred_text == 'Normal', 0, 1)

            # Isolation Forest
            elif model_type == 'manual_if':
                X_val_processed = preprocessor.transform(X_val)
                y_pred_if = model.predict(X_val_processed)
                y_pred_binary = np.where(y_pred_if == 1, 0, 1) # 1=normal, -1=ataque
                y_pred_text = np.where(y_pred_binary == 1, 'Attack', 'Normal') # IF só prevê binário

            # KMeans / MBKMeans
            elif model_type == 'manual_kmeans':
                with open(paths['threshold'], 'r') as f: threshold = json.load(f)['threshold']
                X_val_processed = preprocessor.transform(X_val)
                distances = model.transform(X_val_processed)
                min_distances = np.min(distances, axis=1)
                y_pred_binary = np.where(min_distances > threshold, 1, 0)
                y_pred_text = np.where(y_pred_binary == 1, 'Attack', 'Normal') # KMeans só prevê binário

            # DBSCAN / HDBSCAN (Re-fit)
            elif model_type == 'manual_cluster':
                print(f"  Aviso: {model_name} requer re-fit nos dados de validação (lógica não supervisionada).")
                print(f"  Usando uma amostra de 25% dos dados de validação para {model_name}...")
                
                X_val_processed = preprocessor.transform(X_val)
                
                # Criar uma amostra estratificada para o re-fit
                sample_indices, _ = train_test_split(range(len(X_val_processed)), train_size=0.25, stratify=y_val_binary_mapped, random_state=42)
                X_val_sample = X_val_processed[sample_indices]
                y_val_sample_bin = y_val_binary_mapped[sample_indices]
                y_val_sample_txt = y_val_text.iloc[sample_indices].values
                
                if model_name == 'DBSCAN':
                    cluster_model = DBSCAN(n_jobs=-1, **paths['model_params'])
                else: # HDBSCAN
                    cluster_model = HDBSCAN(**paths['model_params'])
                
                y_pred_cluster = cluster_model.fit_predict(X_val_sample)
                y_pred_binary = np.where(y_pred_cluster == -1, 1, 0) # -1=ruído (ataque), 0+=cluster (normal)
                y_pred_text = np.where(y_pred_binary == 1, 'Attack', 'Normal')
                
                # Avalia este modelo nos dados de AMOSTRA que ele viu
                calculate_and_store_metrics(model_name, y_val_sample_bin, y_pred_binary, y_val_sample_txt, y_pred_text)
                continue # Pula para o próximo modelo

            else:
                print(f"Tipo de modelo desconhecido: {model_type}")
                continue
                
            # Armazena métricas (para todos os modelos exceto os de cluster, que são tratados acima)
            calculate_and_store_metrics(model_name, y_val_binary_mapped, y_pred_binary, y_val_text, y_pred_text)
            
        except FileNotFoundError as e:
            print(f"  ERRO: Artefato não encontrado. Pulando. Detalhe: {e}")
        except Exception as e:
            print(f"  ERRO ao processar {model_name}. Pulando. Detalhe: {e}")
            import traceback
            traceback.print_exc()

print("\n--- Validação de todos os modelos concluída! ---")

--- Iniciando Loop de Validação dos Modelos ---
False
False
False
False

Validando: RandomForest...
{'type': 'pipeline', 'pipeline': '..\\..\\data\\preprocessors-pipeline\\RF\\rf_pipeline_unsw_nb15.joblib', 'preprocessor': '..\\..\\data\\preprocessors-pipeline\\RF\\rf_preprocessor_unsw_nb15.joblib'}
Calculando métricas para: RandomForest

Validando: XGBoost...
{'type': 'pipeline_xgb', 'pipeline': '..\\..\\data\\preprocessors-pipeline\\XGB\\xgb_pipeline_unsw_nb15.joblib', 'preprocessor': '..\\..\\data\\preprocessors-pipeline\\XGB\\xgb_preprocessor_unsw_nb15.joblib'}
  ERRO ao processar XGBoost. Pulando. Detalhe: 'NoneType' object has no attribute 'inverse_transform'

Validando: SVM...
{'type': 'pipeline_svm', 'pipeline': '..\\..\\data\\preprocessors-pipeline\\SVM\\svm_pipeline_unsw_nb15.joblib', 'preprocessor': '..\\..\\data\\preprocessors-pipeline\\SVM\\svm_preprocessor_unsw_nb15.joblib'}
Tipo de modelo desconhecido: pipeline_svm

Validando: SelfTraining-RF...
{'type': 'manual', 'prepr

Traceback (most recent call last):
  File "C:\Users\GabrielMoreira\AppData\Local\Temp\ipykernel_28860\3999915358.py", line 33, in <module>
    y_pred_text = encoder.inverse_transform(y_pred_encoded)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'inverse_transform'


Calculando métricas para: SelfTraining-RF

Validando: SelfTraining-XGB...
{'type': 'manual_xgb', 'preprocessor': '..\\..\\data\\preprocessors-pipeline\\SelfTraining-XGB\\st_xgb_preprocessor_unsw_nb15.joblib', 'model': '..\\..\\data\\preprocessors-pipeline\\SelfTraining-XGB\\st_xgb_model_unsw_nb15.joblib'}
  ERRO ao processar SelfTraining-XGB. Pulando. Detalhe: 'NoneType' object has no attribute 'inverse_transform'

Validando: LabelSpreading...


Traceback (most recent call last):
  File "C:\Users\GabrielMoreira\AppData\Local\Temp\ipykernel_28860\3999915358.py", line 46, in <module>
    y_pred_text = encoder.inverse_transform(y_pred_encoded)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'inverse_transform'


{'type': 'manual_xgb', 'preprocessor': '..\\..\\data\\preprocessors-pipeline\\LabelSpreading\\ls_preprocessor_unsw_nb15.joblib', 'model': '..\\..\\data\\preprocessors-pipeline\\LabelSpreading\\ls_model_unsw_nb15.joblib', 'encoder': '..\\..\\data\\preprocessors-pipeline\\LabelSpreading\\ls_encoder_unsw_nb15.joblib'}


KeyboardInterrupt: 

In [ ]:
if not all_metrics:
    print("Nenhuma métrica foi calculada. Execute a célula de validação.")
else:
    # Converter a lista de dicionários em um DataFrame
    df_metrics = pd.DataFrame(all_metrics)
    df_metrics = df_metrics.set_index('Modelo')
    
    print("--- Tabela de Comparação de Métricas ---")
    
    # Formatar para exibição
    styled_df = df_metrics.style.format("{:.4f}").background_gradient(cmap='Greens', high=1.0, low=0.0, subset=pd.IndexSlice[:, ['Accuracy', 'F1-Score (Ataque)', 'F1-Score (Normal)']])
    display(styled_df)

In [ ]:
if 'df_metrics' not in locals():
    print("DataFrame de métricas não encontrado. Execute a célula anterior.")
else:
    print("--- Gráficos de Comparação de Métricas ---")
    
    # Lista de métricas para plotar
    metrics_to_plot = ['Accuracy', 'Precision (Ataque)', 'Recall (Ataque)', 'F1-Score (Ataque)']
    
    # Criar subplots
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(20, 16))
    fig.suptitle('Comparação de Desempenho dos Modelos no Dataset Capturado', fontsize=20)
    
    axes_flat = axes.flatten() # Facilita a iteração
    
    for i, metric in enumerate(metrics_to_plot):
        # Ordenar modelos pelo desempenho na métrica atual
        df_sorted = df_metrics[metric].sort_values(ascending=False)
        
        # Plotar
        sns.barplot(x=df_sorted.values, y=df_sorted.index, ax=axes_flat[i], palette='viridis')
        axes_flat[i].set_title(metric, fontsize=16)
        axes_flat[i].set_xlabel('Score', fontsize=12)
        axes_flat[i].set_ylabel('Modelo', fontsize=12)
        axes_flat[i].set_xlim(0, 1.05) # Fixar eixo X de 0 a 1
        
        # Adicionar rótulos de valor nas barras
        for p in axes_flat[i].patches:
            axes_flat[i].annotate(f"{p.get_width():.3f}", 
                                (p.get_width() + 0.01, p.get_y() + p.get_height() / 2),
                                ha='left', va='center', xytext=(5, 0), textcoords='offset points')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

In [ ]:
if not all_confusion_matrices:
    print("Nenhuma matriz de confusão foi gerada. Execute a célula de validação.")
else:
    # Cria o dropdown com os nomes dos modelos
    model_dropdown = widgets.Dropdown(
        options=list(all_confusion_matrices.keys()),
        description='Modelo:',
        value=list(all_confusion_matrices.keys())[0] # Seleciona o primeiro modelo por padrão
    )

    # Cria uma saída para o gráfico
    plot_output = widgets.Output()

    def plot_matrices(model_name):
        """Função chamada quando o dropdown muda."""
        with plot_output:
            clear_output(wait=True) # Limpa a saída anterior
            
            # Pega os dados da matriz de confusão
            cm_data = all_confusion_matrices[model_name]
            bin_cm, bin_labels = cm_data['binary']
            multi_cm, multi_labels = cm_data['multiclass']
            
            # Define o tamanho da figura (larga para duas matrizes)
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 8))
            
            # Plotar Matriz Binária
            disp_bin = ConfusionMatrixDisplay(bin_cm, display_labels=bin_labels)
            disp_bin.plot(ax=ax1, cmap='Blues', xticks_rotation=0)
            ax1.set_title(f'{model_name} - Matriz Binária', fontsize=16)
            
            # Plotar Matriz Multiclasse
            disp_multi = ConfusionMatrixDisplay(multi_cm, display_labels=multi_labels)
            disp_multi.plot(ax=ax2, cmap='Greens', xticks_rotation='vertical')
            ax2.set_title(f'{model_name} - Matriz Multiclasse', fontsize=16)
            
            plt.tight_layout()
            plt.show(fig)

    # Conecta a função ao dropdown
    out = widgets.interactive_output(plot_matrices, {'model_name': model_dropdown})
    
    print("--- Visualizador de Matriz de Confusão ---")
    print("Selecione um modelo no dropdown para ver as matrizes de confusão:")
    
    # Exibe o dropdown e a saída do gráfico
    display(VBox([model_dropdown, plot_output]))
    
    # Exibe o primeiro gráfico
    plot_matrices(model_dropdown.value)